# These products have significantly different values at full res compared to 100m.

Issue products:
GMW v3 per ACSC2: https://csdr.dev.oceandevelopmentdata.org/console/product/cacf0035-ec1d-49c2-8edb-031a88323ba5
GMW v4 per ACSC2: https://csdr.dev.oceandevelopmentdata.org/console/product/dd484b2c-c8cf-4c3f-af34-993cd6ef18f6

## GMW v4 per ACSC2
### Peninsula Range - 2020
10m: 173,100,800 m²
https://csdr.dev.oceandevelopmentdata.org/console/product/output/21b92b36-aab9-4a3b-8fa2-d6e5e909f7e1
100m: 55,550,000 m²
https://csdr.dev.oceandevelopmentdata.org/console/product/output/cd419b4c-6863-4566-ba16-fcaa4f6f8801
Almost a 3x difference. Most products change only a couple of percent.

## Debugging GMW v4 per ACSC2 for Peninsula Range.

In [6]:
import geopandas as gpd
from shapely.geometry import mapping

geom_file = "/Users/wj/Projects/csdr/csdr-cloud-spatial/cache/geometries/acsc2/0-0-1/runs/acsc2-test-run-id/Australian_Coastal_Sediment_Compartments_-_Secondary_Compartments.parquet"
# Geom native 4283

geom = gpd.read_parquet(geom_file)
geom_pr = geom[geom["csdr-name"] == "Peninsula Range"]
assert len(geom_pr) == 1, (
    f"Expected exactly one geometry for Peninsula Range, but found {len(geom_pr)}."
)

geom_pr_6933 = geom_pr.to_crs("epsg:6933").geometry.values[0]
geom_pr_4326 = geom_pr.to_crs("epsg:4326").geometry.values[0]

print(f"Area in 6933: {geom_pr_6933.area:.2f} m²")

geom_pr_geojson = mapping(geom_pr_4326)
print(geom_pr_geojson)

Area in 6933: 1363418300.61 m²
{'type': 'Polygon', 'coordinates': (((150.6363720700001, -22.18618896499993), (150.6363720700001, -22.188688964999983), (150.6463720700001, -22.188688964999983), (150.6463720700001, -22.19118896499998), (150.6513720700001, -22.19118896499998), (150.6513720700001, -22.19368896499998), (150.6563720700001, -22.19368896499998), (150.6563720700001, -22.198688964999974), (150.65887207000003, -22.198688964999974), (150.65887207000003, -22.20118896499997), (150.6563720700001, -22.20118896499997), (150.6563720700001, -22.20368896499997), (150.65387207000003, -22.20368896499997), (150.65387207000003, -22.206188964999967), (150.64887207000004, -22.206188964999967), (150.64887207000004, -22.20368896499997), (150.6313720700001, -22.20368896499997), (150.6313720700001, -22.206188964999967), (150.6313720700001, -22.211188964999963), (150.63387207000005, -22.211188964999963), (150.63387207000005, -22.21368896499996), (150.6363720700001, -22.21368896499996), (150.63637207

In [7]:
from pystac import ItemCollection
from rustac import search_sync

gmw_v4_file = (
    "/Users/wj/Projects/csdr/csdr-cloud-spatial/cache/datasets/gmw-v4/0-0-1/gmw.parquet"
)
items = search_sync(
    gmw_v4_file,
    intersects=geom_pr_geojson,
    datetime_string_match="2020",
)

print(len(items))
items = ItemCollection(items)

1


In [11]:
from odc.stac import load

results = load(
    items, resolution=100, crs="epsg:6933", geopolygon=geom_pr_4326, chunks={}
)
print(results)

<xarray.Dataset> Size: 405kB
Dimensions:      (y: 644, x: 613, time: 1)
Coordinates:
  * y            (y) float64 5kB -2.762e+06 -2.762e+06 ... -2.826e+06 -2.826e+06
  * x            (x) float64 5kB 1.452e+07 1.452e+07 ... 1.458e+07 1.458e+07
  * time         (time) datetime64[us] 8B 2020-07-02
    spatial_ref  int32 4B 6933
Data variables:
    mangrove     (time, y, x) uint8 395kB dask.array<chunksize=(1, 644, 613), meta=np.ndarray>


In [12]:
data = results["mangrove"]

data = data.where(data == 1.0)

count = float(data.notnull().sum().values)
one_pixel_area_m2 = abs(data.odc.geobox.resolution.x * data.odc.geobox.resolution.y)

total_area_m2 = count * one_pixel_area_m2

print(
    f"Number of pixels classified as mangrove in Peninsula Range in 2020: {count:.0f}"
)
print(f"Total mangrove area in Peninsula Range in 2020: {total_area_m2:.2f} m²")
print(f"Geometry area: {geom_pr_6933.area:.2f} m²")

print(f"Mangrove coverage: {round((total_area_m2 / geom_pr_6933.area) * 100, 2)}%")

Number of pixels classified as mangrove in Peninsula Range in 2020: 17420
Total mangrove area in Peninsula Range in 2020: 174200000.00 m²
Geometry area: 1363418300.61 m²
Mangrove coverage: 12.78%


At 100m loading I get `174,200,000.00 m²`
At 10m loading I get  `173,100,800.00 m²`

Something has changed in the code/data/geometry because the difference above is expected, not the `55,550,000 m²` that was calculated earlier.

The strange thing is that all of the other products are stable. What has changed for these 2?